# Phase 4 — Future Withdrawal Target Creation and Labeling
### Problem Statement ID 26184: Predictive Analytics Framework for Cybercrime Complaints

## 1. Objective
Construct a leakage-safe binary classification target `future_withdrawal` defining whether a qualifying physical cash withdrawal occurs within the next 24 hours at or near the victim's reported location (same district or <= 10 km radius).


## 2. Load Phase 3 Dataset
Loading the 68-column feature-engineered dataset.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROC_DIR = Path('../data/processed')
df_feat = pd.read_csv(PROC_DIR / 'feature_engineered_cybercrime_data.csv')
print(f'Loaded Feature Matrix: {df_feat.shape[0]:,} rows x {df_feat.shape[1]} columns')
df_feat.head(2)


## 3. Identify Withdrawal Events
Loading the 80,000 physical ATM cashout records.


In [ ]:
withd = pd.read_csv(PROC_DIR / 'cleaned_withdrawals.csv')
print(f'Total ATM Withdrawals: {len(withd):,}')
print(f'Case-Linked Withdrawals: {withd["case_id"].notna().sum():,}')
print(f'Background Normal Withdrawals: {withd["case_id"].isna().sum():,}')
withd.head(2)


## 4. Identify Timestamps
Verifying chronological fields for complaints (T0) and withdrawals (Tw).


In [ ]:
ts_def = pd.read_csv('../outputs/phase4_timestamp_definition.csv')
ts_def


## 5. Identify Location Representation
Reviewing coordinates and administrative district levels.


In [ ]:
print('Complaint Locations sample:')
print(df_feat[['victim_area', 'victim_district', 'latitude', 'longitude']].head(3))
print('\nWithdrawal Locations sample:')
print(withd[['area', 'district', 'latitude', 'longitude', 'atm_id']].head(3))


## 6. Define 24-Hour Prediction Window
Enforcing strict forward-looking constraints (T0 < Tw <= T0 + 24h).


In [ ]:
# Prediction Window Rule:
# 1. delay_hours > 0.0 (strictly future; no retrospective or simultaneous events)
# 2. delay_hours <= 24.0 (strictly bounded to the 24h intervention horizon)
print('Prediction Horizon: 24.0 Hours')


## 7. Create Future Withdrawal Target
Executing spatio-temporal matching using vectorized Haversine distance.


In [ ]:
c_df = df_feat[['case_id', 'complaint_timestamp', 'latitude', 'longitude', 'victim_area_id', 'victim_district']].copy()
c_df['dt_c'] = pd.to_datetime(c_df['complaint_timestamp'])

w_df = withd[['case_id', 'timestamp', 'latitude', 'longitude', 'area_id', 'district', 'atm_id']].dropna(subset=['case_id']).copy()
w_df['dt_w'] = pd.to_datetime(w_df['timestamp'])

merged = pd.merge(c_df, w_df, on='case_id', suffixes=('_c', '_w'))
merged['delay_hours'] = (merged['dt_w'] - merged['dt_c']).dt.total_seconds() / 3600.0

lat1, lon1 = np.radians(merged['latitude_c']), np.radians(merged['longitude_c'])
lat2, lon2 = np.radians(merged['latitude_w']), np.radians(merged['longitude_w'])
dlat = lat2 - lat1
dlon = lon2 - lon1
a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
merged['dist_km'] = (6371.0 * 2 * np.arcsin(np.sqrt(a))).round(2)

qualifying = merged[
    (merged['delay_hours'] > 0.0) &
    (merged['delay_hours'] <= 24.0) &
    ((merged['victim_district'] == merged['district']) | (merged['dist_km'] <= 10.0))
]
pos_cases = set(qualifying['case_id'].unique())
df_feat['future_withdrawal'] = df_feat['case_id'].isin(pos_cases).astype(int)
print(df_feat['future_withdrawal'].value_counts())


## 8. Leakage Prevention
Isolating target and outcome metadata from candidate model feature matrix.


In [ ]:
leakage_df = pd.read_csv('../outputs/target_leakage_columns.csv')
leakage_df


## 9. Target Validation
Running automated sanity checks on target values and boundaries.


In [ ]:
val_report = pd.read_csv('../outputs/target_validation_report.csv')
val_report


## 10. Class Distribution
Analyzing positive and negative outcome frequency.


In [ ]:
dist_df = pd.read_csv('../outputs/future_withdrawal_class_distribution.csv')
display(dist_df)
from IPython.display import Image
Image(filename='../outputs/figures/future_withdrawal_distribution.png')


## 11. Temporal Validation
Verifying observation horizon completeness across all records.


In [ ]:
max_w = pd.to_datetime(withd['timestamp']).max()
df_feat['target_observation_complete'] = ((pd.to_datetime(df_feat['complaint_timestamp']) + pd.Timedelta(hours=24)) <= max_w).astype(int)
print(f'Observation Complete Records: {df_feat["target_observation_complete"].sum():,} / {len(df_feat):,} (100%)')


## 12. Manual Verification
Inspecting 20 sample case audits with exact coordinates and temporal deltas.


In [ ]:
manual_df = pd.read_csv('../outputs/target_manual_verification.csv')
manual_df[['case_id', 'time_difference_hours', 'spatial_distance_km', 'future_withdrawal', 'explanation']]


## 13. Save Targeted Dataset
Exporting labeled dataset to data/processed/targeted_cybercrime_data.csv.


In [ ]:
df_feat.to_csv(PROC_DIR / 'targeted_cybercrime_data.csv', index=False)
print(f'Exported Targeted Dataset: {df_feat.shape[0]:,} rows x {df_feat.shape[1]} cols')


## 14. Limitations
- The target specifically isolates local cashouts (10.27%); distant interstate mule cashouts (mean 383km) are labeled 0 for local intervention.
- Unlinked background civilian withdrawals are excluded from fraud labeling.


## 15. Next Phase
**PHASE 5 — TEMPORAL TRAIN/VALIDATION/TEST SPLIT**
Chronological time-series splitting to guarantee zero future-leakage in model evaluation.
